# 02 — Data Preprocessing

This notebook prepares the raw review dataset for machine learning.

The main objectives are to inspect duplicate reviews, validate the target labels,
perform necessary cleaning, and create training and testing datasets while
preventing data leakage.

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split


# Load dataset
df = pd.read_csv("../data/raw/fake reviews dataset.csv")

# Display first 5 rows
df.head()


print("Original dataset shape:", df.shape)

# Verify that the required columns are present
required_columns = {"text_", "label", "category", "rating"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

Original dataset shape: (40432, 4)


In [10]:
df_processed = df.copy()

# Check for missing review texts and labels
missing_texts = df_processed["text_"].isna().sum()
missing_labels = df_processed["label"].isna().sum()

print("Missing review texts:", missing_texts)
print("Missing labels:", missing_labels)

assert missing_texts == 0, "Missing review texts found."
assert missing_labels == 0, "Missing labels found."

# Remove leading and trailing whitespace
df_processed["text_"] = (
    df_processed["text_"]
    .astype("string")
    .str.strip()
)

# Check for empty review texts after normalization
empty_texts = df_processed["text_"].eq("").sum()

print("Empty review texts after cleaning:", empty_texts)

assert empty_texts == 0, "Empty review texts found after cleaning."

# Count duplicate review texts after normalization
duplicate_count = df_processed["text_"].duplicated().sum()

print("Duplicate review texts to remove:", duplicate_count)

# Check whether duplicate review texts have conflicting labels
label_counts = (
    df_processed.groupby("text_")["label"]
    .nunique()
)

conflicting_reviews = int((label_counts > 1).sum())

print("Unique review texts with conflicting labels:", conflicting_reviews)

assert conflicting_reviews == 0, (
    "Identical review texts with conflicting labels were found."
)

# Remove duplicate review texts
df_processed = df_processed.drop_duplicates(
    subset=["text_"],
    keep="first"
).reset_index(drop=True)

print("Dataset shape after duplicate removal:", df_processed.shape)

Missing review texts: 0
Missing labels: 0
Empty review texts after cleaning: 0
Duplicate review texts to remove: 22
Unique review texts with conflicting labels: 0
Dataset shape after duplicate removal: (40410, 4)


In [11]:
label_mapping = {
    "OR": 0,
    "CG": 1
}

df_processed["target"] = df_processed["label"].map(label_mapping)

# Verify that every label was mapped
if df_processed["target"].isna().any():
    unknown_labels = df_processed.loc[
        df_processed["target"].isna(), "label"
    ].unique()

    raise ValueError(f"Unknown labels found: {unknown_labels}")

df_processed["target"] = df_processed["target"].astype(int)

print("Label mapping:")
display(df_processed[["label", "target"]].drop_duplicates())

print("\nTarget distribution:")
print(df_processed["target"].value_counts().sort_index())

print("\nTarget distribution (percentage):")
print((df_processed["target"].value_counts(normalize=True).sort_index() * 100).round(2))

Label mapping:


,label,target
0,CG,1
55,OR,0



Target distribution:
target
0    20215
1    20195
Name: count, dtype: int64

Target distribution (percentage):
target
0    50.02
1    49.98
Name: proportion, dtype: float64


In [12]:
# Features and target
X = df_processed["text_"]
y = df_processed["target"]

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting label distribution:")
print(y_test.value_counts().sort_index())

print("\nTraining label distribution (%):")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))

print("\nTesting label distribution (%):")
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))

Training samples: 32328
Testing samples: 8082

Training label distribution:
target
0    16172
1    16156
Name: count, dtype: int64

Testing label distribution:
target
0    4043
1    4039
Name: count, dtype: int64

Training label distribution (%):
target
0    50.02
1    49.98
Name: proportion, dtype: float64

Testing label distribution (%):
target
0    50.02
1    49.98
Name: proportion, dtype: float64


In [13]:
# Save train and test datasets
train_df = pd.DataFrame({
    "text": X_train,
    "target": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "target": y_test
})

train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

print("Training and testing datasets saved successfully.")

Training and testing datasets saved successfully.


In [14]:
# Check for missing values
print("Missing values in training data:")
print(train_df.isnull().sum())

print("\nMissing values in testing data:")
print(test_df.isnull().sum())

# Check for duplicate reviews within each dataset
train_duplicates = train_df["text"].duplicated().sum()
test_duplicates = test_df["text"].duplicated().sum()

print("\nDuplicate training reviews:", train_duplicates)
print("Duplicate testing reviews:", test_duplicates)

# Check for overlap between training and testing reviews
train_texts = set(train_df["text"])
test_texts = set(test_df["text"])

overlap = train_texts.intersection(test_texts)

print("\nReviews appearing in both sets:", len(overlap))

# Validate the saved datasets
assert train_df.isnull().sum().sum() == 0, "Missing values in training data."
assert test_df.isnull().sum().sum() == 0, "Missing values in testing data."
assert train_duplicates == 0, "Duplicate reviews in training data."
assert test_duplicates == 0, "Duplicate reviews in testing data."
assert len(overlap) == 0, "Train-test overlap detected."

print("\nAll preprocessing validation checks passed.")

Missing values in training data:
text      0
target    0
dtype: int64

Missing values in testing data:
text      0
target    0
dtype: int64

Duplicate training reviews: 0
Duplicate testing reviews: 0

Reviews appearing in both sets: 0

All preprocessing validation checks passed.


### Handling Duplicate Reviews

The original dataset contained 20 duplicate review-text occurrences. After removing leading and trailing whitespace, this increased to 22 duplicate occurrences.

We removed repeated review texts while retaining one occurrence of each unique review. This decision was made for two reasons:

1. To prevent identical reviews from appearing in both training and testing sets, which could lead to data leakage and overly optimistic evaluation results.
2. To prevent repeated records from giving certain review texts disproportionate influence during model training.

Some duplicated review texts had different ratings, but none had conflicting CG/OR labels. Since our primary classification task uses review text rather than ratings, we retained the first occurrence of each unique review text.